# qwen21-nsfw-pose · Colab T4 运行笔记本

> 运行前：菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**。
>
> ⚠️ 未在 T4 上实测；内存只有 12.7GB，文本编码器 8.7GB 可能 OOM。详见仓库 `install/colab.md`。
>
> **Drive 用法**：
> - 输出图、LoRA 放 Drive（持久化）。
> - 底模约 16GB，默认缓存到 Drive（`CACHE_BASE_MODELS_ON_DRIVE = True`）；本地已有的底模会自动补存。
> - 免费 Drive 只有 15GB 装不下，请改成 False，每次新会话从 HuggingFace 重下。
> - 想要一键版，用 `colab_t4_quick.ipynb`。
> - 模型会先复制到本地盘再加载，不要直接从 Drive 加载（FUSE 读取慢）。

In [ ]:
# ===== 1. 配置 =====
import os

REPO_URL = "https://github.com/ldybyz/qwen21-nsfw-pose-skill.git"
BRANCH = "feat/colab-t4-guide"  # 合并进 main 后可改回 main

USE_DRIVE = True
CACHE_BASE_MODELS_ON_DRIVE = True  # 底模约 16GB；免费 Drive(15GB) 装不下时改成 False
DRIVE_ROOT = "/content/drive/MyDrive/qwen21_pose"

COMFY = "/content/ComfyUI"
POSELAB = "/content/qwen21-nsfw-pose-skill"
LORA_NAME = "qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors"

# LoRA 获取方式二选一：
#  A) 手动下载后上传到 Drive 的 qwen21_pose/loras/（文件名必须等于 LORA_NAME）
#  B) 填下面两项自动下载：在 Civitai 模型页对应版本的 Download 按钮上复制链接
#     （形如 https://civitai.com/api/download/models/<版本ID>），token 在 Civitai 账号设置里生成
LORA_URL = ""
CIVITAI_TOKEN = ""

os.environ.update(
    COMFY=COMFY,
    POSELAB=POSELAB,
    COMFYUI_URL="http://127.0.0.1:8188",
    COMFYUI_INPUT=f"{COMFY}/input",
    COMFYUI_OUTPUT=f"{COMFY}/output",
)

In [ ]:
# ===== 2. 挂载 Google Drive =====
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for d in ("loras", "models", "outputs"):
        os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
    print("Drive 目录：", DRIVE_ROOT)
    print("请把 LoRA .safetensors 放进：", f"{DRIVE_ROOT}/loras")

In [ ]:
# ===== 3. 克隆仓库与 ComfyUI，安装依赖 =====
!test -d {POSELAB} || git clone -b {BRANCH} {REPO_URL} {POSELAB}
!cd {POSELAB} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!test -d {COMFY} || git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY}
!cd {COMFY} && git pull -q && pip install -q -r requirements.txt

In [ ]:
# ===== 4. 底模：优先从 Drive 复制，否则从 HuggingFace 下载 =====
import shutil, subprocess

HF = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main"
BASE = [
    ("diffusion_models", "qwen_image_2.1_int8_convrot.safetensors"),
    ("text_encoders", "qwen3vl_8b_int8_convrot.safetensors"),
    ("vae", "qwen_image_2.1_vae_bf16.safetensors"),
]
for sub, name in BASE:
    dst_dir = f"{COMFY}/models/{sub}"
    os.makedirs(dst_dir, exist_ok=True)
    dst = f"{dst_dir}/{name}"
    drv = f"{DRIVE_ROOT}/models/{sub}/{name}"
    if os.path.exists(dst):
        # 本地已有但 Drive 里没有 → 补存到 Drive
        if USE_DRIVE and CACHE_BASE_MODELS_ON_DRIVE and not os.path.exists(drv):
            os.makedirs(os.path.dirname(drv), exist_ok=True)
            shutil.copyfile(dst, drv + ".part")
            os.replace(drv + ".part", drv)
            print("已缓存到 Drive：", name)
        else:
            print("已存在：", name)
        continue
    if USE_DRIVE and os.path.exists(drv):
        print("从 Drive 复制：", name)
        shutil.copyfile(drv, dst)
    else:
        print("从 HuggingFace 下载：", name)
        subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", dst, f"{HF}/{sub}/{name}"], check=True)
        if USE_DRIVE and CACHE_BASE_MODELS_ON_DRIVE:
            os.makedirs(os.path.dirname(drv), exist_ok=True)
            shutil.copyfile(dst, drv)
            print("已缓存到 Drive：", name)

In [ ]:
# ===== 5. LoRA：从 Drive 复制到 ComfyUI =====
lora_dst = f"{COMFY}/models/loras"
os.makedirs(lora_dst, exist_ok=True)
if USE_DRIVE:
    src_dir = f"{DRIVE_ROOT}/loras"
    for f in os.listdir(src_dir):
        if f.endswith(".safetensors") and not os.path.exists(f"{lora_dst}/{f}"):
            shutil.copyfile(f"{src_dir}/{f}", f"{lora_dst}/{f}")
            print("已复制 LoRA：", f)
else:
    print("未使用 Drive：请手动把 LoRA 上传到", lora_dst)

# 缺少 LoRA 且填了下载链接 → 自动下载（用 -O 强制保存为 LORA_NAME），并缓存到 Drive
if LORA_NAME not in os.listdir(lora_dst) and LORA_URL:
    cmd = ["wget", "-q", "--show-progress", "-O", f"{lora_dst}/{LORA_NAME}"]
    if CIVITAI_TOKEN:
        cmd += ["--header", f"Authorization: Bearer {CIVITAI_TOKEN}"]
    subprocess.run(cmd + [LORA_URL], check=True)
    if USE_DRIVE:
        shutil.copyfile(f"{lora_dst}/{LORA_NAME}", f"{DRIVE_ROOT}/loras/{LORA_NAME}")

have = [f for f in os.listdir(lora_dst) if f.endswith(".safetensors")]
print("当前 LoRA：", have)
if LORA_NAME not in have:
    raise RuntimeError(
        f"缺少 {LORA_NAME}。\n"
        f"方式A：把文件放进 Drive 的 {DRIVE_ROOT}/loras/（文件名必须完全一致），再重跑本 cell；\n"
        "方式B：在第 1 个 cell 填 LORA_URL / CIVITAI_TOKEN，重跑第 1 个和本 cell。\n"
        "（put_loras_here 只是 ComfyUI 的占位文件，不是 LoRA）"
    )

In [ ]:
# ===== 6. 后台启动 ComfyUI =====
import time, urllib.request

log = open(f"{COMFY}/comfy.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188",
     "--lowvram", "--disable-smart-memory", "--fp32-vae"],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT,
)

ready = False
for _ in range(120):
    if proc.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3)
        ready = True
        break
    except Exception:
        time.sleep(3)
print("ComfyUI 就绪" if ready else "启动失败，请查看日志（下一个 cell）")

In [ ]:
# ===== 7. （排错）查看 ComfyUI 日志末尾 =====
!tail -n 60 {COMFY}/comfy.log

In [ ]:
# ===== 8. 环境自检 =====
!cd {POSELAB} && python3 scripts/doctor.py

In [ ]:
# ===== 9. 出图并保存到 Drive =====
# 先用小参数冒烟测试（512x512 / 10 步）；确认无黑图、无 OOM 后，再逐步加到 1024x1024 / 30 步。
# 注意：相对 README 配方（1152x1728 / 50 步）是降配，交付时需如实标注。
WIDTH, HEIGHT, STEPS = 512, 512, 10
SEED = 80964

import datetime
from IPython.display import Image, display

prompt = open(f"{POSELAB}/prompts/taws_v13_shower.txt", encoding="utf-8").read()
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out = f"{POSELAB}/outputs/colab_{stamp}.png"

r = subprocess.run(
    ["python3", "scripts/qwen_pose_test.py", "--mode", "t2i", "--name", "colab_test",
     "--prompt", prompt, "--seed", str(SEED), "--steps", str(STEPS),
     "--cfg", "2.5", "--sampler", "er_sde", "--scheduler", "beta",
     "--width", str(WIDTH), "--height", str(HEIGHT),
     "--lora", LORA_NAME, "--lora-strength", "1.0", "--out", out],
    cwd=POSELAB,
)

if r.returncode == 0 and os.path.exists(out):
    if USE_DRIVE:
        shutil.copy(out, f"{DRIVE_ROOT}/outputs/")
        shutil.copy(out + ".json", f"{DRIVE_ROOT}/outputs/")
        print("已保存到 Drive：", f"{DRIVE_ROOT}/outputs/")
    display(Image(out))
else:
    print("生成失败，查看上方报错及 ComfyUI 日志")